##**Libraries & Dependencies**

In [35]:
import pandas as pd
import numpy as np
import os
import sys

In [36]:
import librosa
import librosa.display
import seaborn as sns
import matplotlib.pyplot as plt

In [37]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

In [38]:
import IPython.display as ipd
from IPython.display import Audio
import keras
from keras.preprocessing import sequence
from keras.models import Sequential
from keras.layers import Dense, Embedding
from keras.layers import LSTM,BatchNormalization , GRU
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from keras.layers import Input, Flatten, Dropout, Activation
from keras.layers import Conv1D, MaxPooling1D, AveragePooling1D
from keras.models import Model
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import SGD

In [39]:
import warnings
if not sys.warnoptions:
    warnings.simplefilter("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
import tensorflow as tf
print ("Done")

Done


##**Dataset paths**

In [40]:
ravdess = "/kaggle/input/ravdess-emotional-speech-audio"

Crema = "/kaggle/input/cremad/AudioWAV/"

Tess = "/kaggle/input/toronto-emotional-speech-set-tess/tess toronto emotional speech set data/TESS Toronto emotional speech set data/"

Savee = "/kaggle/input/surrey-audiovisual-expressed-emotion-savee/ALL/"

# We will use multiple datasets for more data and wider range

##**Collecting Datasets**

1. Ravdess

In [41]:
import kagglehub

# Download latest version
ravdess_download_path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")

print("Path to dataset files:", ravdess_download_path)

Using Colab cache for faster access to the 'ravdess-emotional-speech-audio' dataset.
Path to dataset files: /kaggle/input/ravdess-emotional-speech-audio


In [42]:
paths = []
emotions = []

emotion_dict = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

# Use the downloaded path directly
for root, _, files in os.walk(ravdess_download_path):
    for file in files:
        if file.endswith('.wav'):
            # Extract emotion from filename: e.g., '03-01-04-01-02-01-01.wav'
            # The emotion code is the third part of the filename (index 2 after splitting by '-')
            emotion_code = file.split('-')[2]
            if emotion_code in emotion_dict:
                emotion = emotion_dict[emotion_code]
                paths.append(os.path.join(root, file))
                emotions.append(emotion)

# Create a DataFrame
df = pd.DataFrame({
    'path': paths,
    'emotion': emotions
})

display(df.head())

,path,emotion
0,/kaggle/input/ravdess-emotional-speech-audio/A...,surprised
1,/kaggle/input/ravdess-emotional-speech-audio/A...,neutral
2,/kaggle/input/ravdess-emotional-speech-audio/A...,disgust
3,/kaggle/input/ravdess-emotional-speech-audio/A...,disgust
4,/kaggle/input/ravdess-emotional-speech-audio/A...,neutral


In [43]:
print(df.emotion.value_counts())

emotion
surprised    384
disgust      384
fearful      384
sad          384
happy        384
calm         384
angry        384
neutral      192
Name: count, dtype: int64


2. Crema

In [44]:

# Download latest version
crema_download_path = kagglehub.dataset_download("ejlok1/cremad")

print("Path to dataset files:", crema_download_path)

Using Colab cache for faster access to the 'cremad' dataset.
Path to dataset files: /kaggle/input/cremad


In [45]:
crema_paths = []
crema_emotions = []

# Use the downloaded path and the 'AudioWAV' subdirectory
path_crema_data = os.path.join(crema_download_path, 'AudioWAV')

# Emotion mapping for Crema dataset based on file naming convention
crema_emotion_dict = {
    'ANG': 'angry',
    'DIS': 'disgust',
    'FEA': 'fearful',
    'HAP': 'happy',
    'NEU': 'neutral',
    'SAD': 'sad'
    # The dataset description also mentions 'SUR' for surprised, but based on the provided subset and common usage,
    # I'll stick to these main ones. If 'SUR' files are present, they will not be mapped without adding 'SUR': 'surprised'.
}

for root, _, files in os.walk(path_crema_data):
    for file in files:
        if file.endswith('.wav'):
            # Extract emotion from filename: e.g., '1001_DFA_ANG_XX.wav'
            # The emotion code is the third part of the filename (index 2 after splitting by '_')
            parts = file.split('_')
            if len(parts) > 2:
                emotion_code = parts[2]
                if emotion_code in crema_emotion_dict:
                    emotion = crema_emotion_dict[emotion_code]
                    crema_paths.append(os.path.join(root, file))
                    crema_emotions.append(emotion)

# Create a DataFrame for Crema
df_crema = pd.DataFrame({
    'path': crema_paths,
    'emotion': crema_emotions
})

display(df_crema.head())

,path,emotion
0,/kaggle/input/cremad/AudioWAV/1028_TSI_DIS_XX.wav,disgust
1,/kaggle/input/cremad/AudioWAV/1075_IEO_HAP_LO.wav,happy
2,/kaggle/input/cremad/AudioWAV/1084_ITS_HAP_XX.wav,happy
3,/kaggle/input/cremad/AudioWAV/1067_IWW_DIS_XX.wav,disgust
4,/kaggle/input/cremad/AudioWAV/1066_TIE_DIS_XX.wav,disgust


In [46]:
print(df_crema.emotion.value_counts())

emotion
disgust    1271
happy      1271
sad        1271
fearful    1271
angry      1271
neutral    1087
Name: count, dtype: int64


3. Tess

In [47]:


# Download latest version
tess_download_path = kagglehub.dataset_download("ejlok1/toronto-emotional-speech-set-tess")

print("Path to dataset files:", tess_download_path)

Using Colab cache for faster access to the 'toronto-emotional-speech-set-tess' dataset.
Path to dataset files: /kaggle/input/toronto-emotional-speech-set-tess


In [ ]:
tess_paths = []
tess_emotions = []

# Use the downloaded path and the 'TESS Toronto emotional speech set data' subdirectory
path_tess_data = os.path.join(tess_download_path, 'TESS Toronto emotional speech set data')

for root, _, files in os.walk(path_tess_data):
    for file in files:
        if file.endswith('.wav'):
            # Extract emotion from filename: e.g., 'OAF_sad.wav'
            # The emotion is typically the part before the '.wav' extension, and after the first underscore.
            parts = file.split('_')
            if len(parts) > 1:
                emotion_raw = parts[-1].replace('.wav', '')
                emotion = emotion_raw.lower() # Standardize to lowercase
                tess_paths.append(os.path.join(root, file))
                tess_emotions.append(emotion)

# Create a DataFrame for Tess
df_tess = pd.DataFrame({
    'path': tess_paths,
    'emotion': tess_emotions
})

display(df_tess.head())

In [ ]:
print(df_tess.emotion.value_counts())

4. Savee Dataset

In [ ]:


# Download latest version
savee_download_path = kagglehub.dataset_download("ejlok1/surrey-audiovisual-expressed-emotion-savee")

print("Path to dataset files:", savee_download_path)

In [ ]:
savee_paths = []
savee_emotions = []

# Use the downloaded path and the 'ALL' subdirectory
path_savee_data = os.path.join(savee_download_path, 'ALL')

# Emotion mapping for Savee dataset based on file naming convention
# Example filenames: 'DC_a01.wav', 'JE_sa02.wav'
savee_emotion_dict = {
    'a': 'angry',
    'd': 'disgust',
    'f': 'fearful',
    'h': 'happy',
    'n': 'neutral',
    'sa': 'sad',
    'su': 'surprised'
}

for root, _, files in os.walk(path_savee_data):
    for file in files:
        if file.endswith('.wav'):
            # Extract emotion from filename: e.g., 'DC_a01.wav', 'JE_sa01.wav'
            # The emotion code is typically after the underscore and before the number.
            # Example: 'DC_a01.wav' -> parts[1] = 'a01' -> emotion_code = 'a'
            # Example: 'JE_sa01.wav' -> parts[1] = 'sa01' -> emotion_code = 'sa'
            parts = file.split('_')
            if len(parts) > 1:
                emotion_raw = parts[1].split('.')[0] # Remove .wav extension
                emotion_code = ''.join(filter(str.isalpha, emotion_raw)) # Keep only alphabetic characters

                if emotion_code in savee_emotion_dict:
                    emotion = savee_emotion_dict[emotion_code]
                    savee_paths.append(os.path.join(root, file))
                    savee_emotions.append(emotion)

# Create a DataFrame for Savee
df_savee = pd.DataFrame({
    'path': savee_paths,
    'emotion': savee_emotions
})

display(df_savee.head())

In [ ]:
print(df_savee.emotion.value_counts())

##**Concatinating Datasets**

In [ ]:
super_dataframe = pd.concat([df, df_crema, df_tess, df_savee], axis=0)
super_dataframe.reset_index(drop=True, inplace=True)
display(super_dataframe.head())

In [ ]:
super_dataframe.shape

In [ ]:
super_dataframe.emotion.value_counts()

In [ ]:
plt.title('Count of Emotions', size=16)
sns.countplot(super_dataframe.emotion)
plt.ylabel('Count', size=12)
plt.xlabel('Emotions', size=12)
sns.despine(top=True, right=True, left=False, bottom=False)
plt.show()


##**Analyzing Audio**

In [ ]:
data,sr = librosa.load(savee_paths[1000])
sr

In [ ]:
ipd.Audio(data,rate=sr)

In [ ]:
plt.figure(figsize=(10, 5))
spectrogram = librosa.feature.melspectrogram(y=data, sr=sr, n_mels=128,fmax=8000)
log_spectrogram = librosa.power_to_db(spectrogram)
librosa.display.specshow(log_spectrogram, y_axis='mel', sr=sr, x_axis='time');
plt.title('Mel Spectrogram ')
plt.colorbar(format='%+2.0f dB')

Horizontal axis represents time in seconds while vertical axis shoes the frequency Range

In [ ]:
mfcc = librosa.feature.mfcc(y=data, sr=sr, n_mfcc=30)


# MFCC
plt.figure(figsize=(16, 10))
plt.subplot(3,1,1)
librosa.display.specshow(mfcc, x_axis='time')
plt.ylabel('MFCC')
plt.colorbar()

ipd.Audio(data,rate=sr)

##**Data Augmentation**

In [ ]:
def add_noise(data):
    noise_amp = 0.035*np.random.uniform()*np.amax(data)
    data = data + noise_amp*np.random.normal(size=data.shape[0])
    return data

In [ ]:
def add_shift(audio, sr=None):  # Add second parameter
    shift_range = int(np.random.uniform(low=-5, high=5) * 1000)
    augmented_audio = np.roll(audio, shift_range)
    return augmented_audio

In [ ]:
def stretch(data, rate=0.8):
    return librosa.effects.time_stretch(data, rate=rate)

def add_pitch(data, sampling_rate, pitch_factor=0.7):
    return librosa.effects.pitch_shift(data, sr=sampling_rate, n_steps=pitch_factor)


Testing

In [ ]:
plt.figure(figsize=(12, 5))
librosa.display.waveshow(y=data, sr=sr)
ipd.Audio(data,rate=sr)

In [ ]:
x = add_noise(data)
plt.figure(figsize=(12,5))
librosa.display.waveshow(y=x, sr=sr)
ipd.Audio(x, rate=sr)

In [ ]:
x = add_pitch(data, sr)
plt.figure(figsize=(12,5))
librosa.display.waveshow(y=x, sr=sr)
ipd.Audio(x, rate=sr)

In [ ]:
x = add_shift(data, sr)
plt.figure(figsize=(12,5))
librosa.display.waveshow(y=x, sr=sr)
ipd.Audio(x, rate=sr)

In [ ]:
x = add_shift(data, sr)
plt.figure(figsize=(12,5))
librosa.display.waveshow(y=x, sr=sr)
ipd.Audio(x, rate=sr)

##**Feature Extration**

In [ ]:
def zcr(data,frame_length,hop_length):
    zcr=librosa.feature.zero_crossing_rate(data,frame_length=frame_length,hop_length=hop_length)
    return np.squeeze(zcr)

In [ ]:
def rmse(data,frame_length=2048,hop_length=512):
    rmse=librosa.feature.rms(y=data,frame_length=frame_length,hop_length=hop_length)
    return np.squeeze(rmse)

In [ ]:
def calculate_mfcc(data,sr,frame_length=2048,hop_length=512,flatten:bool=True):
    # Ensuring librosa.feature.mfcc is called with explicit 'y' argument.
    mfcc_features=librosa.feature.mfcc(y=data,sr=sr)
    return np.squeeze(mfcc_features.T)if not flatten else np.ravel(mfcc_features.T)

In [ ]:
def extract_features(data,sr=22050,frame_length=2048,hop_length=512):
    result=np.array([])

    # Using the updated calculate_mfcc function.
    result=np.hstack((
                      zcr(data,frame_length,hop_length),
                      rmse(data,frame_length,hop_length),
                      calculate_mfcc(data,sr,frame_length,hop_length)
                     ))
    return result

In [ ]:
def get_features(path,duration=2.5, offset=0.6):
    data,sr=librosa.load(path,duration=duration,offset=offset)
    aud=extract_features(data)
    audio=np.array(aud)

    noised_audio=add_noise(data)
    aud2=extract_features(noised_audio)
    audio=np.vstack((audio,aud2))

    pitched_audio=add_pitch(data,sr)
    aud3=extract_features(pitched_audio)
    audio=np.vstack((audio,aud3))

    pitched_audio1=add_pitch(data,sr)
    pitched_noised_audio=add_noise(pitched_audio1)
    aud4=extract_features(pitched_noised_audio)
    audio=np.vstack((audio,aud4))

    return audio

In [ ]:
!pip install tqdm_joblib

In [ ]:
from joblib import Parallel, delayed
import timeit
from tqdm import tqdm

start = timeit.default_timer()

# Define a function to get features for a single audio file
def process_feature(path, emotion):
    features = get_features(path)
    X = []
    Y = []
    for ele in features:
        X.append(ele)
        # appending emotion 3 times as we have made 3 augmentation techniques on each audio file.
        Y.append(emotion)
    return X, Y

paths = super_dataframe.path
emotions = super_dataframe.emotion

# Progress bar for parallel processing
print(f"Processing {len(paths)} audio files...")


from tqdm_joblib import tqdm_joblib


with tqdm_joblib(tqdm(desc="Processing audio files", total=len(paths))) as progress_bar:
    results = Parallel(n_jobs=-1)(
        delayed(process_feature)(path, emotion)
        for (path, emotion) in zip(paths, emotions)
    )

# Collect the results
X = []
Y = []
for result in results:
    x, y = result
    X.extend(x)
    Y.extend(y)

stop = timeit.default_timer()

print(f'Time: {stop - start:.2f} seconds')
print(f'Processed {len(X)} feature vectors from {len(paths)} audio files')

##**Making Dataframe**

In [ ]:
len(X), len(Y), super_dataframe.path.shape

In [ ]:
processed_data = pd.DataFrame(X)
processed_data['emotion'] = Y
processed_data.to_csv('emotion.csv', index=False)
processed_data.head()

In [ ]:
Emotions = pd.read_csv('./emotion.csv')

Cleaning Data

In [ ]:
print(processed_data.isna().any())

In [ ]:
processed_data=processed_data.fillna(0)
print(processed_data.isna().any())
processed_data.shape

Downloading Direct from google drive to same time

In [ ]:
import gdown

# Define the file ID from the provided URL
file_id = '1zNQjFZ6lQKUK4HA7r-VIOymut42mnnwm'

# Define the output file name
output_file = 'stock_market_data.csv'

# Download the file
gdown.download(id=file_id, output=output_file, quiet=False)

# Load the dataset into a pandas DataFrame
df = pd.read_csv(output_file)

# Display the first 5 rows of the DataFrame
print(f"Successfully loaded '{output_file}' with {len(df)} rows and {len(df.columns)} columns.")
display(df.head())

##**Data Prepration**

In [ ]:
X = df.iloc[: ,:-1].values
Y = df['emotion'].values

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
encoder = OneHotEncoder()
Y = encoder.fit_transform(np.array(Y).reshape(-1,1)).toarray()

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, Y, random_state=42,test_size=0.2, shuffle=True)
x_train.shape, y_train.shape, x_test.shape, y_test.shape

In [ ]:
X_train = x_train.reshape(x_train.shape[0] , x_train.shape[1] , 1)
X_test = x_test.reshape(x_test.shape[0] , x_test.shape[1] , 1)

In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
x_train.shape, y_train.shape, x_test.shape, y_test.shape

##**Model Training**

In [ ]:
import keras
from keras.preprocessing import sequence
from keras.models import Sequential
from keras.layers import Dense, Embedding
from keras.layers import LSTM,BatchNormalization , GRU
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from keras.layers import Input, Flatten, Dropout, Activation
from keras.layers import Conv1D, MaxPooling1D, AveragePooling1D
from keras.models import Model
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import SGD
import tensorflow as tf
from tensorflow.keras.layers import Dropout

In [ ]:
x_traincnn =np.expand_dims(x_train, axis=2)
x_testcnn= np.expand_dims(x_test, axis=2)
x_traincnn.shape, y_train.shape, x_testcnn.shape, y_test.shape

In [ ]:
import tensorflow.keras.layers as L

model = tf.keras.Sequential([
    L.Conv1D(512,kernel_size=5, strides=1,padding='same', activation='relu',input_shape=(X_train.shape[1],1)),
    L.BatchNormalization(),
    L.MaxPool1D(pool_size=5,strides=2,padding='same'),

    L.Conv1D(512,kernel_size=5,strides=1,padding='same',activation='relu'),
    L.BatchNormalization(),
    L.MaxPool1D(pool_size=5,strides=2,padding='same'),
    Dropout(0.2),  # Add dropout layer after the second max pooling layer

    L.Conv1D(256,kernel_size=5,strides=1,padding='same',activation='relu'),
    L.BatchNormalization(),
    L.MaxPool1D(pool_size=5,strides=2,padding='same'),

    L.Conv1D(256,kernel_size=3,strides=1,padding='same',activation='relu'),
    L.BatchNormalization(),
    L.MaxPool1D(pool_size=5,strides=2,padding='same'),
    Dropout(0.2),  # Add dropout layer after the fourth max pooling layer

    L.Conv1D(128,kernel_size=3,strides=1,padding='same',activation='relu'),
    L.BatchNormalization(),
    L.MaxPool1D(pool_size=3,strides=2,padding='same'),
    Dropout(0.2),  # Add dropout layer after the fifth max pooling layer

    L.Flatten(),
    L.Dense(512,activation='relu'),
    L.BatchNormalization(),
    L.Dense(10,activation='softmax')
])
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
model.summary()

In [ ]:
early_stop=EarlyStopping(monitor='val_acc',mode='min',patience=5,restore_best_weights=True)
lr_reduction=ReduceLROnPlateau(monitor='val_acc',patience=3,verbose=1,factor=0.5,min_lr=0.00001, mode = 'min')

In [ ]:
history = model.fit(
    x_traincnn,
    y_train,
    batch_size=64,
    epochs=5,
    validation_data=(x_testcnn, y_test),
    callbacks=[early_stop, lr_reduction]
)

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')

##**Dumping Model**

In [ ]:
import joblib

# Save the CNN model
model.save('cnn_model.h5')
print("CNN model saved as cnn_model.h5")

# Save the StandardScaler
joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved as scaler.pkl")

# Save the OneHotEncoder
joblib.dump(encoder, 'encoder.pkl')
print("Encoder saved as encoder.pkl")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Saving Models to Google Drive

Now, let's save the trained CNN model, the StandardScaler, and the OneHotEncoder to Google Drive for persistent storage and easy access for deployment.

In [ ]:
import shutil

# Define the destination path in Google Drive
drive_path = '/content/drive/MyDrive/emotion_recognition_model/'

# Create the directory if it doesn't exist
os.makedirs(drive_path, exist_ok=True)

print(f"Google Drive path set to: {drive_path}")

In [ ]:
# List of files to copy
files_to_copy = ['cnn_model.h5', 'scaler.pkl', 'encoder.pkl']

for file_name in files_to_copy:
    source_path = file_name
    destination_path = os.path.join(drive_path, file_name)

    try:
        shutil.copy(source_path, destination_path)
        print(f"Successfully copied '{file_name}' to '{destination_path}'")
    except FileNotFoundError:
        print(f"Error: '{file_name}' not found. Please ensure it was saved correctly.")
    except Exception as e:
        print(f"An error occurred while copying '{file_name}': {e}")

### Deployment Pipeline

To ensure consistent predictions in a deployment environment (like Flask), we need to load the saved model, feature scaler, and emotion encoder. We'll then create a function that replicates the feature extraction, scaling, and prediction steps used during training.

In [ ]:
import tensorflow as tf
import joblib
import numpy as np
import librosa
import os

# Define the path to your saved models in Google Drive
drive_path = '/content/drive/MyDrive/emotion_recognition_model/'

# Load the saved model
try:
    loaded_model = tf.keras.models.load_model(os.path.join(drive_path, 'cnn_model.h5'))
    print("CNN model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")

# Load the saved scaler
try:
    loaded_scaler = joblib.load(os.path.join(drive_path, 'scaler.pkl'))
    print("Scaler loaded successfully!")
except Exception as e:
    print(f"Error loading scaler: {e}")

# Load the saved encoder
try:
    loaded_encoder = joblib.load(os.path.join(drive_path, 'encoder.pkl'))
    print("Encoder loaded successfully!")
except Exception as e:
    print(f"Error loading encoder: {e}")


#### Prediction Function

This function will take an audio file path, extract its features using the same methods as during training, apply the loaded scaler, and then use the loaded CNN model to predict the emotion. Finally, it will decode the one-hot encoded prediction back to a human-readable emotion label.

In [ ]:
def predict_emotion(audio_path, model, scaler, encoder, duration=2.5, offset=0.6, sr=22050):
    # 1. Load and extract features from the audio file
    data, _ = librosa.load(audio_path, duration=duration, offset=offset, sr=sr)
    features = extract_features(data, sr=sr)

    # Reshape for scaler if needed (if extract_features returns a 1D array)
    if features.ndim == 1:
        features = features.reshape(1, -1)

    # 2. Scale the features using the loaded scaler
    scaled_features = scaler.transform(features)

    # 3. Reshape for the CNN model (add channel dimension)
    # The CNN model expects input shape (batch_size, num_features, 1)
    reshaped_features = np.expand_dims(scaled_features, axis=2)

    # 4. Make prediction with the loaded model
    prediction = model.predict(reshaped_features)

    # 5. Decode the prediction using the loaded encoder
    # The prediction will be a probability distribution over classes.
    # Get the index of the highest probability.
    predicted_class_idx = np.argmax(prediction, axis=1)

    # Create a dummy one-hot array to inverse transform
    one_hot_prediction = np.zeros((1, len(encoder.categories_[0])))
    one_hot_prediction[0, predicted_class_idx] = 1

    predicted_emotion = encoder.inverse_transform(one_hot_prediction)[0][0]

    return predicted_emotion, prediction


#### Demonstration

Let's test the `predict_emotion` function with one of the audio files from our dataset. We'll pick a file and compare the predicted emotion with its actual emotion.

In [ ]:
# Get a sample audio file path and its true emotion from the original dataframe
sample_audio_path = super_dataframe['path'].iloc[0]
sample_true_emotion = super_dataframe['emotion'].iloc[0]

print(f"Sample audio path: {sample_audio_path}")
print(f"True emotion for sample: {sample_true_emotion}")

# Make a prediction
predicted_emotion, probabilities = predict_emotion(sample_audio_path, loaded_model, loaded_scaler, loaded_encoder)

print(f"Predicted emotion: {predicted_emotion}")
print(f"Prediction probabilities: {probabilities}")

# You can play the audio to verify
print("Playing sample audio:")
Audio(sample_audio_path)
